# 03 — Feature engineering, temporal validation, and training

This notebook turns the municipality-month table from `02` into the model inputs,
splits the series in time, and trains the seven predictors compared in the paper.

**Seven predictors.** Three baselines and four learned models. *Persistence*
predicts that the target month repeats the outbreak state of the row's own month
t. That state is not yet consolidated when a forecast is issued (cases take about
60 days to settle), so *real-time persistence* repeats the state of month t-1,
the last consolidated one. *Climatology* predicts each municipality's historical
outbreak rate for the calendar month of the target, t+h, fitted on training rows
only. The learned models are logistic regression, random forest, XGBoost and
LightGBM. All of them see the same
features, the same folds and the same labels, so the comparison is like for like.

**Expanding-window validation.** To score 2022 the model trains on everything
whose target falls before 2022; to score 2023, everything before 2023; and so on.
Nothing from the future ever enters training.

**Four leakage guards** are described where they are applied: the fold boundary
follows the month being predicted rather than the month observed; rolling windows
are built with a group-wise transform so no window crosses between
municipalities; every rolling feature is shifted before it is aggregated; and no
feature derives from a label whose threshold was fitted over the full series.

Outputs land in `data/train/`: one metrics file and one prediction file per
disease, definition and horizon, plus fitted models for the subset listed in
`SAVE_MODELS_FOR`.


## Section 0 — Setup: load master + labels

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
MASTER_DIR = PROJECT_ROOT / "data" / "master"
OUT_DIR = PROJECT_ROOT / "data" / "train"
OUT_DIR.mkdir(parents=True, exist_ok=True)

X_master = pd.read_parquet(MASTER_DIR / "municipio_mes.parquet")
y_labels = pd.read_parquet(MASTER_DIR / "labels.parquet")

df = X_master.merge(y_labels, on=["cod_ibge", "ano", "mes"], how="left")
df = df.sort_values(["cod_ibge", "ano", "mes"]).reset_index(drop=True)
print(f"Merged: {df.shape}")

## Section 1 — Build features (leakage-safe)

Every feature at *t* is computed from data available at *t* and no later. Three
rules do the work, and each exists because its absence is a specific way to leak:

- **Lags use `shift(k)`** within the municipality, so a value never crosses from
  one municipality into another.
- **Rolling means and trends use `shift(1)` before the window**, so the window
  ends at *t-1* and never contains the month being described. They also run
  through `transform`, which keeps the window inside the municipality — applying
  `.rolling()` to a flat shifted series pulls values across that boundary.
- **The lagged endemic-channel label is deliberately excluded.** It reads as a
  harmless "was there an outbreak last month", but its threshold was fitted over
  the whole series, so the value encodes years that come after the row. The
  reasoning and the measurement are in the comment where it would have been built.

One naming quirk of this master to keep in mind: yellow-fever cases use the `fa_`
prefix while its SIH hospitalizations use `febre_amarela_`.

In [ ]:
DISEASES = ["dengue", "zika", "chikungunya", "fa"]
CLIMATE_VARS = ["t2m", "prectotcorr", "rh2m", "ws10m", "ps"]
LAGS_CASES = [1, 2, 3, 6, 12]
ROLL_WINDOWS = [3, 6]

# Fill case NaN -> 0 (no notification = zero inside the grid) BEFORE any grouping,
# and add incidence + past-outbreak columns to df, so a single groupby sees them all.
for d in DISEASES:
    df[f"{d}_casos"] = df[f"{d}_casos"].fillna(0)
    df[f"{d}_incid"] = df[f"{d}_casos"] / df["pop_residente"] * 100_000

# NOTE — the lagged endemic-channel label is deliberately NOT used as a feature.
# It looks harmless (a past outbreak is observable at the time), but the canal
# label's threshold is median + 1.96*sigma over the NON-EPIDEMIC years of the
# whole series, and for dengue that set includes 2022 and 2023 — test years.
# Recomputing the label with a baseline restricted to years before 2022 flips
# 2,722 of 54,180 pre-test labels (5.0%), across 525 of the 645 municipalities:
# 4,200 pre-test outbreaks become 6,494. So the value a model would see in 2018
# encodes how bad 2022-2023 turned out. That is lookahead, and it goes out.
# The label itself stays as it is: as a TARGET the fixed window is a documented,
# defensible choice — it is only its use as an INPUT that leaks.

# SIH hospitalizations: fill 0; yellow fever uses 'febre_amarela' prefix
SIH_COL = {"dengue": "dengue_internacoes_sih", "zika": "zika_internacoes_sih",
           "chikungunya": "chikungunya_internacoes_sih", "fa": "febre_amarela_internacoes_sih"}
for d in DISEASES:
    if SIH_COL[d] in df.columns:
        df[SIH_COL[d]] = df[SIH_COL[d]].fillna(0)

# ONE groupby, created AFTER all df columns exist -> no stale-groupby leakage
g = df.groupby("cod_ibge", group_keys=False)

def lag(col, k):
    return g[col].shift(k)

def roll_mean(col, w):
    # Two guards, both necessary.
    # shift(1) first: the window ends at t-1 and never includes t.
    # transform: the rolling window must stay INSIDE the municipality. Calling
    # .rolling() on g[col].shift(1) rolls over the FLAT series and pulls values
    # across the boundary between municipalities — measured on this base: 1,148
    # rows, every one of them the first or second month of a municipality.
    return g[col].transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())

def trend3(col):
    def slope(x):
        y = x.values
        m = ~np.isnan(y)
        if m.sum() < 2:
            return np.nan
        t = np.arange(len(y))
        return np.polyfit(t[m], y[m], 1)[0]
    # transform, for the same reason as roll_mean: the window must not reach
    # back into the previous municipality.
    return g[col].transform(
        lambda s: s.shift(1).rolling(3, min_periods=2).apply(slope, raw=False))

feats = df[["cod_ibge", "ano", "mes"]].copy()

# --- per-disease: case lags/rolls, incidence lags/rolls, trend, past outbreaks ---
for d in DISEASES:
    for k in LAGS_CASES:
        feats[f"{d}_casos_lag{k}"] = lag(f"{d}_casos", k)
    for w in ROLL_WINDOWS:
        feats[f"{d}_casos_roll{w}"] = roll_mean(f"{d}_casos", w)
    for k in [1, 2, 3]:
        feats[f"{d}_incid_lag{k}"] = lag(f"{d}_incid", k)
    for w in ROLL_WINDOWS:
        feats[f"{d}_incid_roll{w}"] = roll_mean(f"{d}_incid", w)
    feats[f"{d}_casos_trend3"] = trend3(f"{d}_casos")

# --- climate: current month + lag1/lag2 + roll3 ---
for v in CLIMATE_VARS:
    feats[v] = df[v].values
    feats[f"{v}_lag1"] = lag(v, 1)
    feats[f"{v}_lag2"] = lag(v, 2)
    feats[f"{v}_roll3"] = roll_mean(v, 3)

# --- seasonality (cyclic) ---
feats["mes_sin"] = np.sin(2 * np.pi * feats["mes"] / 12)
feats["mes_cos"] = np.cos(2 * np.pi * feats["mes"] / 12)

# --- operational-delay sources: lag1 ---
for d in DISEASES:
    if SIH_COL[d] in df.columns:
        feats[f"{SIH_COL[d]}_lag1"] = lag(SIH_COL[d], 1)
for d in ["dengue", "zika", "chikungunya"]:
    for stem in (f"{d}_latencia_mediana", f"{d}_latencia_p90"):
        if stem in df.columns:
            feats[f"{stem}_lag1"] = lag(stem, 1)
for c in ("esf_cobertura_pct", "esf_qt_equipes"):
    if c in df.columns:
        feats[f"{c}_lag1"] = lag(c, 1)

# --- static / annual: pass through ---
STATIC_ANNUAL = [
    "pop_residente", "pib_per_capita", "densidade_hab_km2", "area_km2",
    "gini_2010", "idhm_2010", "rmm_por_100k_nv",
    "num_favelas", "pop_favelas", "cob_vac_fa_pct", "pendulares_saem_trabalho",
    "iag0001_atend_agua_pct", "ies0001_atend_esgoto_pct", "ies0007_esgoto_tratado_pct",
    "iap0001_vias_pavimentadas_pct",
    "pct_floresta", "pct_agricultura", "pct_nao_vegetado", "pct_agua", "pct_natural_nao_florestal",
    "msau28_pacs", "msau541_vig_sanitaria", "msau542_vig_epidemiologica", "msau543_controle_endemias",
    "mgrd01_seca", "mgrd06_alagamento", "mgrd07_erosao", "mgrd08_enchente_gradual",
    "mgrd11_enxurrada", "mgrd14_deslizamento", "mgrd201_mapeamento_risco", "mmam2612_moradia_risco",
    "esf_pop_referencia",
]
for c in STATIC_ANNUAL:
    if c in df.columns:
        feats[c] = df[c].values

# --- categoricals -> one-hot ---
onehot_frames = []
for cat in ("capag", "esf_metodologia"):
    if cat in df.columns:
        onehot_frames.append(pd.get_dummies(df[cat], prefix=cat, dummy_na=False).astype(int))
if onehot_frames:
    feats = pd.concat([feats.reset_index(drop=True)] + [f.reset_index(drop=True) for f in onehot_frames], axis=1)

print(f"Features: {feats.shape}")
all_nan = [c for c in feats.columns if feats[c].isna().all()]
print("All-NaN columns:", all_nan if all_nan else "none")
feats.head(3)

In [ ]:
# Convert boolean-like columns (True/False, possibly with NaN) to numeric float.
# XGBoost accepts float with NaN (treats NaN as missing), but rejects object dtype.
for c in feats.columns:
    if feats[c].dtype == "object" or feats[c].dtype == "bool":
        # map True/False to 1.0/0.0, keep NaN as NaN
        feats[c] = feats[c].map({True: 1.0, False: 0.0}).astype("float32")

# confirm no more object columns
obj_cols = [c for c in feats.columns if feats[c].dtype == "object"]
print("Object columns remaining:", obj_cols if obj_cols else "none")
print("dtypes sample:", feats[["mgrd01_seca", "msau28_pacs", "mmam2612_moradia_risco"]].dtypes.to_dict())

## Section 2 — Shifted target per horizon h

In [ ]:
def build_target(labels_df, disease, definition, h):
    # outbreak at t+h, per municipality; last h months become NaN (dropped later)
    col = f"{disease}_{definition}"
    return (labels_df.sort_values(["cod_ibge", "ano", "mes"])
                     .groupby("cod_ibge")[col].shift(-h))

_t = build_target(y_labels, "dengue", "inc100", 1)
print(f"dengue inc100 h=1: {_t.notna().sum():,} non-null, {int(_t.sum()):,} outbreaks")

## Section 3 — Expanding-window temporal split

In [ ]:
TEST_YEARS = [2022, 2023, 2024]

def add_target_year(frame, h):
    """Year of the outbreak being predicted, h months ahead of the observation.

    The fold split keys on the year of the TARGET, not of the observation: a row
    belongs in training only if the outbreak it predicts happened before the test
    year. The obvious `ano + (mes == 12)` is correct ONLY for h == 1, because only
    December crosses the year boundary one month ahead. At h == 6, months 7 to 11
    also land in the next year — and with the naive formula those rows keep the
    previous year, so they train on labels from the test period while rows late in
    the test year are scored against outcomes from the year after it.
    """
    frame = frame.copy()
    total = frame["ano"] * 12 + (frame["mes"] - 1) + h
    frame["target_year"] = total // 12
    return frame

def expanding_folds(frame, test_years=TEST_YEARS):
    for ty in test_years:
        yield (ty,
               frame.index[frame["target_year"] < ty],
               frame.index[frame["target_year"] == ty])

# Fold sizes shift with the horizon, precisely because the target year does.
for h_demo in (1, 6):
    print(f"h={h_demo}:")
    for ty, tr, te in expanding_folds(add_target_year(feats.copy(), h_demo)):
        print(f"  test {ty}: train={len(tr):,}  test={len(te):,}")

## Section 4 — Validate one combination before scaling

Train dengue/inc100/RF/h=1 on the last fold before running the whole grid, as a
smoke test on the plumbing rather than a result.

What to look for is the *order of magnitude*, not a target number. AUROC in the
high 0.8s to low 0.9s on this task is what a sound pipeline produces. **A value
near 0.99 means something leaked** — that is the signal to stop and find it, not
to celebrate.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

disease, definition, h = "dengue", "inc100", 1

work = feats.copy()
work["target"] = build_target(y_labels, disease, definition, h).values
work = add_target_year(work, h).dropna(subset=["target"])
work["target"] = work["target"].astype(int)

NON_FEATURES = ["cod_ibge", "ano", "mes", "target", "target_year"]
feat_cols = [c for c in work.columns if c not in NON_FEATURES]

ty = 2024
tr = work.index[work["target_year"] < ty]
te = work.index[work["target_year"] == ty]
X_train, y_train = work.loc[tr, feat_cols].fillna(0), work.loc[tr, "target"]
X_test,  y_test  = work.loc[te, feat_cols].fillna(0), work.loc[te, "target"]

print(f"Train {X_train.shape}, pos {100*y_train.mean():.1f}%  |  Test {X_test.shape}, pos {100*y_test.mean():.1f}%")

rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
proba = rf.predict_proba(X_test)[:, 1]
print(f"\ndengue inc100 h=1, test 2024:")
auroc = roc_auc_score(y_test, proba)
print(f"  AUROC {auroc:.3f}")
print(f"  AUPRC {average_precision_score(y_test, proba):.3f}   "
      f"(chance level = prevalence {y_test.mean():.3f})")
if auroc > 0.98:
    print("\n  WARNING: AUROC above 0.98 on this task means something leaked."
          "\n  Stop and audit the features before running the full grid.")

In [ ]:
# Compare our feature set against the old pipeline's expected features
NON_FEATURES = ["cod_ibge", "ano", "mes"]
our_feats = set(c for c in feats.columns if c not in NON_FEATURES)
print(f"Our feature count: {len(our_feats)}")

# list our features grouped by type, to eyeball what we have
import re
print("\nOur climate features:", sorted([c for c in our_feats if any(v in c for v in ['t2m','prectotcorr','rh2m','ws10m','ps'])]))
print("\nOur one-hot (capag/esf_metodologia):", sorted([c for c in our_feats if c.startswith(('capag','esf_metodologia'))]))
print("\nCoordinates present?", [c for c in our_feats if c in ('lat','lon','dist_estacao_km')])

## Section 5 — Training loop (paper subset, checkpointed)

Filled together after Section 4 validates. Trains all diseases × definitions × learned models + three baselines × 6 horizons × 3 folds, saving predictions and metrics to disk as each finishes.

In [ ]:
import time
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb

# ===============================================================
# Section 5 - DEFINITIVE training pipeline
#   Full grid: 4 learned models + 3 baselines x 4 diseases x 4 definitions
#   x 6 horizons x 3 folds.
#   Saves: metrics (results/), per-sample predictions (predictions/),
#          and every trained model (models/).
#   Priority order: dengue first, then zika/chik/fa, so the dengue numbers are
#   saved early and you can start 04 while the rest finishes.
#   Checkpointed: re-running skips combos already on disk.
# ===============================================================
RESULTS_DIR = OUT_DIR / "results"
PRED_DIR    = OUT_DIR / "predictions"
MODELS_DIR  = OUT_DIR / "models"
for d in (RESULTS_DIR, PRED_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

FEAT_COLS_EXCLUDE = ["cod_ibge", "ano", "mes", "target", "target_year"]

# Model artifacts are kept only for these combos — the ones being inspected for
# the paper. Metrics and predictions are still written for every combo; this only
# avoids putting ~3.6 GB of .joblib on disk.
SAVE_MODELS_FOR = {
    ("dengue", "inc100", 1), ("dengue", "inc100", 6),
    ("dengue", "canal", 1), ("dengue", "canal", 6),
    ("chikungunya", "canal", 1), ("chikungunya", "canal", 6),
    ("zika", "canal", 1), ("zika", "canal", 6),
}

def model_factories():
    def rf():   return RandomForestClassifier(n_estimators=200, min_samples_leaf=20,
                                              class_weight="balanced", n_jobs=-1, random_state=42)
    def lgbm(): return lgb.LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.05,
                                          class_weight="balanced", n_jobs=-1, verbose=-1, random_state=42)
    def xgbm(): return xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                         subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                                         n_jobs=-1, eval_metric="aucpr", random_state=42)
    def logreg(): return Pipeline([("imp", SimpleImputer(strategy="median")),
                                   ("sc", StandardScaler()),
                                   ("clf", LogisticRegression(class_weight="balanced",
                                                              max_iter=2000, random_state=42))])
    # Models are fitted per fold; fitted artifacts are kept only for the combos in
    # SAVE_MODELS_FOR.
    return {"rf": rf, "lgbm": lgbm, "xgb": xgbm, "logreg": logreg}

def baseline_persistence(work_te, disease, definition):
    col = f"{disease}_{definition}"
    return work_te.merge(y_labels[["cod_ibge","ano","mes",col]],
                         on=["cod_ibge","ano","mes"], how="left")[col].fillna(0).values.astype(float)

def baseline_persistence_tm1(work_te, disease, definition):
    """Real-time persistence: the outbreak state of month t-1.

    On the issue date (the start of month t+2) month t-1 is the last one whose
    cases have had 60 days to consolidate. `baseline_persistence` uses month t,
    which the models never see and which is not yet consolidated then.
    """
    col = f"{disease}_{definition}"
    nxt = y_labels[["cod_ibge", "ano", "mes", col]].copy()
    idx = nxt["ano"] * 12 + (nxt["mes"] - 1) + 1      # month t-1 attached to month t
    nxt["ano"], nxt["mes"] = idx // 12, idx % 12 + 1
    return work_te.merge(nxt, on=["cod_ibge", "ano", "mes"],
                         how="left")[col].fillna(0).values.astype(float)

def target_month(frame, h):
    """Calendar month (1-12) of the target, h months after the row's own month."""
    return (frame["ano"] * 12 + (frame["mes"] - 1) + h) % 12 + 1

def baseline_climatology(work_tr, work_te, h):
    """Historical outbreak rate of the municipality in the calendar month of the TARGET.

    Fitted on the training targets only, so it uses outbreak labels from years
    before the test year. It keys on the month being predicted (t+h): keying on
    the row's own month t would score the outbreak at t+h with the rate of a
    month h months earlier in the season.
    """
    tr = work_tr.assign(mes_alvo=target_month(work_tr, h))
    freq = tr.groupby(["cod_ibge", "mes_alvo"])["target"].mean().rename("clim")
    te = work_te.assign(mes_alvo=target_month(work_te, h))
    return te.merge(freq, on=["cod_ibge", "mes_alvo"], how="left")["clim"].fillna(0).values

def prepare_work(disease, definition, h):
    work = feats.copy()
    work["target"] = build_target(y_labels, disease, definition, h).values
    work = add_target_year(work, h).dropna(subset=["target"])
    work["target"] = work["target"].astype(int)
    feat_cols = [c for c in work.columns if c not in FEAT_COLS_EXCLUDE]
    return work, feat_cols

def score(yte, proba):
    if yte.nunique() < 2:
        return float("nan"), float("nan")
    return roc_auc_score(yte, proba), average_precision_score(yte, proba)

def run_combo(disease, definition, h, save_models=True):
    work, feat_cols = prepare_work(disease, definition, h)
    factories = model_factories()
    rows, preds = [], []
    for ty, tr_idx, te_idx in expanding_folds(work):
        if len(te_idx) == 0 or work.loc[tr_idx, "target"].sum() == 0:
            continue
        Xtr, ytr = work.loc[tr_idx, feat_cols].fillna(0), work.loc[tr_idx, "target"]
        Xte, yte = work.loc[te_idx, feat_cols].fillna(0), work.loc[te_idx, "target"]
        meta_te  = work.loc[te_idx, ["cod_ibge","ano","mes"]]
        probas = {}
        for name, factory in factories.items():
            try:
                mdl = factory()
                if name == "xgb":
                    npos = int(ytr.sum()); nneg = len(ytr)-npos
                    mdl.set_params(scale_pos_weight=(nneg/max(npos,1)))
                mdl.fit(Xtr, ytr)
                probas[name] = mdl.predict_proba(Xte)[:,1]
                if save_models:
                    fn = MODELS_DIR / f"{disease}_{definition}_h{h}_{name}_{ty}.joblib"
                    joblib.dump(mdl, fn, compress=3)
            except Exception as e:
                print(f"      [skip {name} {ty}] {e}")
        probas["persistence"] = baseline_persistence(work.loc[te_idx], disease, definition)
        probas["persistence_tm1"] = baseline_persistence_tm1(work.loc[te_idx], disease, definition)
        probas["climatology"] = baseline_climatology(work.loc[tr_idx], work.loc[te_idx], h)
        for name, proba in probas.items():
            au, ap = score(yte, proba)
            rows.append({"disease":disease,"definition":definition,"h":h,"test_year":ty,
                         "model":name,"auroc":au,"auprc":ap,"n_test":len(yte),"n_pos":int(yte.sum())})
            p = meta_te.copy()
            p["disease"]=disease; p["definition"]=definition; p["h"]=h
            p["model"]=name; p["test_year"]=ty; p["y_true"]=yte.values; p["proba"]=proba
            preds.append(p)
    return rows, preds

DISEASES_ORDER = ["dengue", "zika", "chikungunya", "fa"]
DEFS_ORDER     = ["inc100", "canal", "inc300", "zscore"]
HORIZONS       = [1, 2, 3, 4, 5, 6]
combos = [(d, defn, h) for d in DISEASES_ORDER for defn in DEFS_ORDER for h in HORIZONS]

t0 = time.time()
for i, (disease, definition, h) in enumerate(combos, 1):
    ckpt = RESULTS_DIR / f"{disease}_{definition}_h{h}.parquet"
    if ckpt.exists():
        print(f"[{i}/{len(combos)}] {disease}/{definition}/h{h}: exists, skip")
        continue
    keep_models = (disease, definition, h) in SAVE_MODELS_FOR
    rows, preds = run_combo(disease, definition, h, save_models=keep_models)
    if rows:
        pd.DataFrame(rows).to_parquet(ckpt)
        pd.concat(preds, ignore_index=True).to_parquet(PRED_DIR / f"{disease}_{definition}_h{h}.parquet")
    print(f"[{i}/{len(combos)}] {disease}/{definition}/h{h}: {len(rows)} results"
          f"{' +models' if keep_models else ''} ({time.time()-t0:.0f}s)")

print(f"\nAll combos done in {time.time()-t0:.0f}s")
print(f"Models saved: {len(list(MODELS_DIR.glob('*.joblib')))}")

In [ ]:
# --- Post-training check ---
# Seven models per combination (rf, lgbm, xgb, logreg, plus the persistence,
# real-time persistence and climatology baselines), three folds, so 21 result
# rows per combo.
# Fitted artifacts are written only for the combos in SAVE_MODELS_FOR, which
# keeps the .joblib total at 8 x 3 folds x 4 trainable models = 96 instead of
# one set per combo.
import pandas as pd

MODELS_PER_COMBO = 7
FOLDS = 3
ROWS_PER_COMBO = MODELS_PER_COMBO * FOLDS

result_files = sorted(RESULTS_DIR.glob("*.parquet"))
print(f"Combos saved: {len(result_files)} of 96")
if not result_files:
    print("Nothing on disk yet - run the training cell above first.")
else:
    todos = pd.concat([pd.read_parquet(f) for f in result_files], ignore_index=True)
    print(f"Result rows: {len(todos):,}")
    print(f"Models:      {sorted(todos['model'].unique())}")
    print(f"Diseases:    {sorted(todos['disease'].unique())}")
    print(f"Definitions: {sorted(todos['definition'].unique())}")

    por_combo = todos.groupby(["disease", "definition", "h"]).size()
    print(f"\nRows per combo - min {por_combo.min()}, max {por_combo.max()}, "
          f"median {int(por_combo.median())}")
    print(f"({ROWS_PER_COMBO} = {MODELS_PER_COMBO} models x {FOLDS} folds; fewer is"
          f" expected for a rare disease, or a fold with no outbreak to score)")

    dengue = todos[todos["disease"] == "dengue"].groupby(["definition", "h"]).size()
    if len(dengue):
        print(f"\nDengue: min {dengue.min()}, max {dengue.max()} "
              f"(expect {ROWS_PER_COMBO} throughout - dengue has outbreaks in every fold)")

# Feature count. This is the guard against silently training on a stale `feats`:
# 125 is the correct number, 137 means the 12 endemic-channel features that leak
# are still in - restart the kernel and rebuild.
n_feats = len([c for c in feats.columns if c not in ("cod_ibge", "ano", "mes")])
print(f"\nFeatures used: {n_feats}")
if n_feats != 125:
    raise RuntimeError(f"expected 125 features, got {n_feats} - stale `feats`, restart the kernel")

saved = list(MODELS_DIR.glob("*.joblib"))
expected_models = len(SAVE_MODELS_FOR) * FOLDS * 4
print(f".joblib saved: {len(saved)} (expected {expected_models} once every combo "
      f"in SAVE_MODELS_FOR has run)")
